# Family-Wise Error Rate (FWER) - Starter Notebook
Demonstrates FWER, Bonferroni correction, and Holm-Bonferroni step-down procedure for multiple hypothesis testing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Simulate m=20 hypothesis tests; first 5 are truly different (H1 true), rest are H0 true
np.random.seed(42)
m = 20
n_per_group = 50
alpha = 0.05

p_values = []
for i in range(m):
    effect = 0.8 if i < 5 else 0.0   # first 5 have real effect
    group1 = np.random.normal(0, 1, n_per_group)
    group2 = np.random.normal(effect, 1, n_per_group)
    _, p = stats.ttest_ind(group1, group2)
    p_values.append(p)

p_values = np.array(p_values)
print(f'Raw p-values (first 5 = true H1):')
for i, p in enumerate(p_values):
    print(f'  Test {i+1:2d}: p={p:.4f}  {"*" if p < alpha else ""}')

In [ ]:
# Bonferroni correction
alpha_bonf = alpha / m
rejected_bonf = p_values < alpha_bonf

# Holm-Bonferroni step-down
order = np.argsort(p_values)
rejected_holm = np.zeros(m, dtype=bool)
for rank, idx in enumerate(order):
    threshold = alpha / (m - rank)
    if p_values[idx] <= threshold:
        rejected_holm[idx] = True
    else:
        break  # stop at first non-rejection

print(f'\nalpha={alpha}, m={m}')
print(f'Bonferroni alpha = {alpha_bonf:.4f}, rejections: {rejected_bonf.sum()}')
print(f'Holm-Bonferroni               rejections: {rejected_holm.sum()}')
print(f'Uncorrected                   rejections: {(p_values < alpha).sum()}')

In [ ]:
# Theoretical FWER: 1 - (1-alpha)^m
ms = np.arange(1, 101)
fwer = 1 - (1 - alpha) ** ms

plt.figure(figsize=(7, 4))
plt.plot(ms, fwer, color='crimson')
plt.axvline(m, color='gray', linestyle='--', label=f'Current m={m}')
plt.xlabel('Number of tests (m)')
plt.ylabel('FWER')
plt.title(f'FWER = 1 - (1 - α)^m  (α={alpha})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot p-values with correction thresholds
sorted_p = np.sort(p_values)
ranks = np.arange(1, m + 1)

plt.figure(figsize=(8, 4))
plt.scatter(ranks, sorted_p, color='steelblue', zorder=3, label='p-values (sorted)')
plt.axhline(alpha, color='gray', linestyle=':', label=f'Uncorrected α={alpha}')
plt.axhline(alpha_bonf, color='red', linestyle='--', label=f'Bonferroni α/m={alpha_bonf:.4f}')
holm_thresholds = [alpha / (m - r) for r in range(m)]
plt.plot(ranks, holm_thresholds, color='green', linestyle='-.',
         label='Holm thresholds')
plt.xlabel('Rank')
plt.ylabel('p-value')
plt.title('Multiple Testing: FWER Corrections')
plt.legend()
plt.tight_layout()
plt.show()